# ThemeCloner V2 — revised point in time walk forward

This notebook keeps the existing data pull and RP PCA modules intact. The revised logic is isolated in three additive modules:

- `src/matching_v3.py`: robust multi ETF factor exposure matching
- `src/walkforward_v3.py`: point in time residualization, RP PCA fitting and basket formation
- `src/evaluation_v3.py`: forward exposure, placebo and semantic validation

The primary objective is to identify small cap stocks with related latent thematic exposures. Raw ETF return tracking is retained only as a secondary diagnostic.

In [1]:
# -------------------- 0. Setup --------------------
import os
import sys

ROOT = os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(ROOT, "config", "etfs.csv")):
    ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print(f"root: {ROOT}")
print(f"config exists: {os.path.exists('config/etfs.csv')}")

root: C:\Users\aamin\ThemeCloner2
config exists: True


## Step 1 — Pull the same raw inputs used by V2

In [2]:
# -------------------- 1a. Parameters and ETF config --------------------
from src.data_pull import load_etf_config, split_themes_controls

START_DATE = "2018-01-01"
K = 15
GAMMA = 10.0
TOP_N = 30
TOP_FACTORS = 3

REBALANCE_FREQ = "Q"
WALKFORWARD_WINDOW = "rolling"        # "rolling" or "expanding"
ROLLING_WINDOW_WEEKS = 208             # four years
MIN_TRAIN_OBS = 104                    # at least two years of valid observations
MIN_TRAIN_COVERAGE = 0.80

INCLUDE_COMMODITY = False
N_PLACEBOS = 50                        # increase to 200+ for final reported results
PLACEBO_ADMISSION_QUANTILE = 0.05
RANDOM_STATE = 42

etf_config = load_etf_config("config/etfs.csv")
themes_config, controls_config = split_themes_controls(etf_config)
print(f"{themes_config['theme'].nunique()} discovery themes")

loaded 44 ETFs: 31 theme-ETFs, 13 control-ETFs (sector/subsector calibration anchors)
  AI Infrastructure: IGPT (IGPT), QTUM (QTUM), SOXX (SOXX)
  Agribusiness: MOO (MOO), VEGI (VEGI), PBJ (PBJ)
  Clean Energy: ICLN (ICLN), QCLN (QCLN), PBW (PBW)
  Critical Minerals: URA (URA), LIT (LIT), COPX (COPX)
  Cybersecurity: CIBR (CIBR), HACK (HACK), BUG (BUG)
  Defense: ITA (ITA), PPA (PPA), XAR (XAR)
  Factor1: IUSG (IUSG)
  Factor2: VLUE (VLUE)
  Factor3: QUAL (QUAL)
  Factor4: SPGP (SPGP)
  Factor5: MTUM (MTUM)
  Factor6: VUG (VUG)
  Factor7: VTV (VTV)
  Fintech: FINX (FINX), ARKF (ARKF), IPAY (IPAY)
  Robotics & AI: BOTZ (BOTZ), ROBO (ROBO), IRBO (IRBO)
  SECTOR: US Energy: XLE (XLE)
  SECTOR: US Financials: XLF (XLF)
  SECTOR: US Technology: XLK (XLK)
  SUBSECTOR: US Banks: KRE (KRE)
  SUBSECTOR: US Biotech: XBI (XBI)
  SUBSECTOR: US Software: IGV (IGV)
  Timber & Forestry: WOOD (WOOD), CUT (CUT)
  US Infrastructure: PAVE (PAVE), IFRA (IFRA)
  Water: PHO (PHO), FIW (FIW), CGW (CGW)
11 di

In [3]:
# -------------------- 1b. ETF returns --------------------
from src.data_pull import pull_etf_returns

etf_returns = pull_etf_returns(etf_config, start=START_DATE)
print(etf_returns.shape)


pulling ETF panel: 44 tickers in 1 batch(es)
  batch 1/1 (44 tickers)...
  dropped 1 tickers below 85% coverage
  ETF panel: 445 weeks x 43 stocks
  date range: 2018-01-12 to 2026-07-17
  ETFs kept: ['ARKF', 'BOTZ', 'CGW', 'CIBR', 'COPX', 'CUT', 'FINX', 'FIW', 'HACK', 'ICLN', 'IFRA', 'IGPT', 'IGV', 'IPAY', 'IRBO', 'ITA', 'IUSG', 'KRE', 'LIT', 'MOO', 'MTUM', 'PAVE', 'PBJ', 'PBW', 'PHO', 'PPA', 'QCLN', 'QTUM', 'QUAL', 'ROBO', 'SOXX', 'SPGP', 'URA', 'VEGI', 'VLUE', 'VTV', 'VUG', 'WOOD', 'XAR', 'XBI', 'XLE', 'XLF', 'XLK']
(445, 43)


In [4]:
# -------------------- 1c. Covariance universe --------------------
from src.data_pull import pull_covariance_universe

cov_returns = pull_covariance_universe(
    start=START_DATE,
    use_cache=True,
    us_only=True,
)
print(cov_returns.shape)

loaded covariance universe from cache (us_only): 1003 tickers across ['us_sp500', 'us_mid_small']

pulling covariance universe: 1003 unique tickers

pulling covariance universe: 1003 tickers in 3 batch(es)
  batch 1/3 (400 tickers)...


$SATS: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)

1 Failed download:
['SATS']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)


  batch 2/3 (400 tickers)...


$BAYA: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)

1 Failed download:
['BAYA']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)


  batch 3/3 (203 tickers)...


$TBH: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)

1 Failed download:
['TBH']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-07-17)


  dropped 251 tickers below 85% coverage
  covariance universe: 445 weeks x 752 stocks
  date range: 2018-01-12 to 2026-07-17
(445, 752)


In [5]:
# -------------------- 1d. Target universe --------------------
from src.data_pull import _cached_returns, pull_target_universe

tgt_returns = _cached_returns(
    "target_universe_returns",
    pull_target_universe,
    refresh=False,
    start=START_DATE,
)
print(tgt_returns.shape)

  loaded target_universe_returns from cache: 444 weeks x 811 cols
(444, 811)


In [6]:
# -------------------- 1e. Observable factor returns --------------------
# IMPORTANT: do not residualize the full sample here.
# The revised walk forward module fits residualization inside each rebalance window.
from src.residualize import get_combined_factors

ff_factors = get_combined_factors(
    start=START_DATE,
    include_commodity=INCLUDE_COMMODITY,
)
print(ff_factors.shape)


pulling FF5 + momentum from Kenneth French library (weekly)...
  momentum fetch failed via pandas_datareader (Unknown datetime string format, unable to parse: Missing data are indicated by -99.99 or -999., at position 0)
  falling back to direct download from Ken French's data library...
  first 20 raw lines (for diagnosis):
    'This file was created by using the 202605 CRSP database.  It,,'
    'contains a momentum factor, constructed from six value-weight portfolios formed,'
    'using independent sorts on size and prior return of NYSE, AMEX, and NASDAQ stocks.'
    'MOM is the average of the returns on two (big and small) high prior return portfolios,,'
    'minus the average of the returns on two low prior return portfolios.  The portfolios,,'
    'are constructed daily.  Big means a firm is above the median market cap on the NYSE,,'
    'at the end of the previous day; small firms are below the median NYSE market cap.,,'
    'Prior return is measured from day - 250 to - 21.  Fir

## Step 2 — Build the point in time rebalance schedule

In [7]:
# -------------------- 2. Rebalance dates --------------------
from src.rppca_walkforward import make_rebalance_dates

# Restrict the schedule to dates where the observable factor panel is available.
model_dates = cov_returns.index.intersection(ff_factors.index)
minimum_window = ROLLING_WINDOW_WEEKS if WALKFORWARD_WINDOW == "rolling" else MIN_TRAIN_OBS

rebalance_dates = make_rebalance_dates(
    model_dates,
    freq=REBALANCE_FREQ,
    min_window_weeks=minimum_window,
)
print(f"{len(rebalance_dates)} rebalance dates")
print(rebalance_dates[:3], "...", rebalance_dates[-3:])

18 rebalance dates
[Timestamp('2021-12-31 00:00:00'), Timestamp('2022-03-25 00:00:00'), Timestamp('2022-06-24 00:00:00')] ... [Timestamp('2025-09-26 00:00:00'), Timestamp('2025-12-26 00:00:00'), Timestamp('2026-03-27 00:00:00')]


## Step 3 — Matching rules

In [8]:
# -------------------- 3. Robust matching configuration --------------------
from src.matching_v3 import MatchingConfig

matching_config = MatchingConfig(
    top_factors=TOP_FACTORS,
    min_etf_r2=0.10,                # theme ETF must be meaningfully explained by the factor space
    min_candidate_r2=0.10,          # candidate loading estimate must be credible
    min_cosine=0.20,                # secondary directional check
    max_relative_distance=None,     # calibrated each period from time shifted placebo themes
    factor_gap_weight=0.25,         # penalize one large factor mismatch
    consensus_weight=0.25,          # penalize disagreement across theme ETFs
    min_etf_matches=1,
    etf_match_distance=None,
)
matching_config

MatchingConfig(top_factors=3, min_etf_r2=0.1, min_candidate_r2=0.1, min_cosine=0.2, max_relative_distance=None, factor_gap_weight=0.25, consensus_weight=0.25, min_etf_matches=1, etf_match_distance=None)

## Step 4 — Run the revised walk forward process

In [9]:
# -------------------- 4. Point in time backtest --------------------
from src.walkforward_v3 import run_point_in_time_backtest

results = run_point_in_time_backtest(
    cov_returns_raw=cov_returns,
    etf_returns_raw=etf_returns,
    target_returns_raw=tgt_returns,
    factor_returns=ff_factors,
    etf_config=themes_config,          # discovery themes only
    rebalance_dates=rebalance_dates,
    K=K,
    gamma=GAMMA,
    window=WALKFORWARD_WINDOW,
    rolling_window_weeks=ROLLING_WINDOW_WEEKS,
    top_n=TOP_N,
    matching_config=matching_config,
    min_train_obs=MIN_TRAIN_OBS,
    min_train_coverage=MIN_TRAIN_COVERAGE,
    n_placebos=N_PLACEBOS,
    placebo_admission_quantile=PLACEBO_ADMISSION_QUANTILE,
    random_state=RANDOM_STATE,
    verbose=True,
)


Rebalance 1/18: 2021-12-31
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      7.050
  top eigenvalues:   [61.8 41.7 25.2 21.2 17.4 16.7 14.9 13.  12.5 11.4 10.2  9.4  9.2  8.8
  8.7]
Selected 317 stocks across 11 themes; forward window has 12 observations.

Rebalance 2/18: 2022-03-25
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      6.540
  top eigenvalues:   [63.7 40.5 25.4 22.6 18.3 16.3 14.8 13.5 12.5 11.4 10.4  9.6  9.   9.
  8.7]
Selected 309 stocks across 11 themes; forward window has 13 observations.

Rebalance 3/18: 2022-06-24
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      6.446
  top eigenvalues:   [62.8 35.6 26.3 22.  18.8 16.6 14.9 13.3 12.4 11.6 10.8  9.7  9.5  9.3
  9. ]
Selected 314 stocks across 11 themes; forward window has 14 observations.

Rebalance 4/18: 2022-09-30
RP-PCA fitted: K=15, gamma=10.0
  in-sample SR:      6.827
  top eigenvalues:   [61.4 39.4 27.7 22.  19.3 16.7 14.5 14.1 12.7 11.7 10.5  9.8  9.7  9.3
  9.1]
Selected 309 stocks across 11 theme

## Step 5 — Review the latest selected baskets

In [10]:
# -------------------- 5a. Latest baskets --------------------
from src.matching_v3 import flatten_baskets

basket_history = flatten_baskets(results["period_baskets"])
latest_date = basket_history["rebalance_date"].max()
latest_baskets = basket_history.loc[basket_history["rebalance_date"] == latest_date]
print(f"latest rebalance: {latest_date.date()}")
latest_baskets.sort_values(["theme", "ticker"])

latest rebalance: 2026-03-27


,rebalance_date,theme,ticker,weight
5127,2026-03-27,AI Infrastructure,ALRS,0.041667
5121,2026-03-27,AI Infrastructure,CWBC,0.041667
5131,2026-03-27,AI Infrastructure,DGICA,0.041667
5132,2026-03-27,AI Infrastructure,EBMT,0.041667
5129,2026-03-27,AI Infrastructure,ELME,0.041667
...,...,...,...,...
5400,2026-03-27,Water,ONIT,0.043478
5391,2026-03-27,Water,PFX,0.043478
5394,2026-03-27,Water,TIPT,0.043478
5403,2026-03-27,Water,TITN,0.043478


In [11]:
# -------------------- 5b. Latest detailed scores --------------------
latest_scores = results["period_scores"][latest_date]
latest_scores.loc[latest_scores["eligible"]].sort_values(
    ["theme", "penalized_distance"]
).groupby("theme", sort=False).head(10)

,ticker,theme,primary_distance,penalized_distance,median_cosine,minimum_cosine,candidate_r2,max_factor_gap,distance_mad,n_etf_matches,n_theme_etfs,eligible,rank_score,medoid_etf,rank
0,TRC,AI Infrastructure,1.319048,1.511227,0.251724,0.176733,0.131309,0.687572,0.081145,2,3,True,-1.511227,IGPT,1
1,MHLA,AI Infrastructure,1.406857,1.608631,0.209268,-0.080963,0.102355,0.714718,0.092378,2,3,True,-1.608631,IGPT,2
2,PFX,AI Infrastructure,1.432041,1.688407,0.395844,0.237783,0.167835,0.983626,0.041837,3,3,True,-1.688407,IGPT,3
3,JRSH,AI Infrastructure,1.592873,1.825456,0.250776,0.121165,0.113745,0.905119,0.025210,2,3,True,-1.825456,IGPT,4
4,FPI,AI Infrastructure,1.748669,2.037961,0.237688,0.219662,0.145094,0.966385,0.190779,3,3,True,-2.037961,IGPT,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8115,HSTM,Water,1.844568,2.079480,0.319003,0.294668,0.189358,0.902455,0.037192,3,3,True,-2.079480,PHO,6
8116,CWCO,Water,1.793061,2.130315,0.542829,0.521684,0.115594,1.328877,0.020137,3,3,True,-2.130315,PHO,7
8117,ISBA,Water,2.072441,2.398347,0.240512,-0.051412,0.104110,1.286322,0.017305,2,3,True,-2.398347,PHO,8
8118,GWRS,Water,2.150645,2.453757,0.413693,0.386986,0.246953,1.183550,0.028898,3,3,True,-2.453757,PHO,9


## Step 6 — Forward exposure evaluation

In [12]:
# -------------------- 6a. Theme level summary --------------------
from src.evaluation_v3 import summarize_forward_evaluation

evaluation_summary = summarize_forward_evaluation(results["evaluation"])
evaluation_summary.sort_values("median_rank_ic", ascending=False)

,theme,periods,median_rank_ic,mean_rank_ic,median_top_bottom_spread,median_selected_forward_corr,median_basket_etf_corr,median_exposure_hit_rate,median_placebo_pvalue,rank_ic_positive_rate
3,Cybersecurity,18,0.116248,0.125904,0.085319,0.107197,0.206615,0.600000,0.375000,0.777778
5,Fintech,18,0.075552,0.090442,0.067745,0.084608,0.022911,0.583333,0.420168,0.722222
9,Water,18,0.065228,0.078378,0.068659,0.037967,0.085747,0.583333,0.604167,0.777778
1,AI Infrastructure,18,0.064491,0.025400,0.035521,0.031158,0.147124,0.528571,0.320513,0.611111
4,Defense,18,0.049492,0.021346,0.044890,0.059008,-0.023344,0.583333,0.207143,0.555556
2,Clean Energy,18,-0.015413,-0.024851,0.010513,0.037299,-0.005093,0.566667,0.162500,0.444444
6,US Infrastructure,18,-0.033095,-0.004847,-0.004087,0.016087,0.139971,0.500000,0.307692,0.444444
0,Robotics & AI,18,-0.038062,-0.036993,-0.026749,0.044972,0.201747,0.533333,0.292857,0.388889
10,Timber & Forestry,18,-0.081820,-0.080865,-0.059748,0.033123,0.283883,0.510870,0.215909,0.111111
8,Critical Minerals,18,-0.083206,-0.085923,-0.063177,0.065810,0.169030,0.633333,0.171569,0.166667


In [13]:
# -------------------- 6b. Period level metrics --------------------
# Primary metrics:
# 1. forward_rank_ic_corr
# 2. top_bottom_corr_spread
# 3. basket_residual_etf_corr
# 4. placebo_pvalue
results["evaluation"].sort_values(["theme", "rebalance_date"])

,rebalance_date,theme,n_selected,forward_rank_ic_beta,forward_rank_ic_corr,top_bottom_beta_spread,top_bottom_corr_spread,selected_mean_forward_beta,selected_mean_forward_corr,exposure_hit_rate,basket_synthetic_beta,basket_synthetic_corr,basket_residual_etf_beta,basket_residual_etf_corr,raw_basket_period_return,n_benchmark_etfs,placebo_pvalue,placebo_median_basket_corr
1,2021-12-31,AI Infrastructure,25,0.108882,0.079507,1.330559,0.056590,0.336177,0.037854,0.560000,0.336177,0.189765,-0.129994,-0.202191,-0.054156,3,0.208333,0.010229
12,2022-03-25,AI Infrastructure,28,0.087791,0.062026,0.246790,0.037885,0.165876,-0.022061,0.464286,0.165876,0.058175,0.187179,0.447899,-0.080089,3,0.363636,-0.134349
23,2022-06-24,AI Infrastructure,14,0.047469,-0.014606,-0.357486,-0.026998,0.517698,0.046805,0.500000,0.517698,0.176555,0.262472,0.415497,-0.070209,3,0.411765,-0.024794
34,2022-09-30,AI Infrastructure,25,0.158039,0.112170,2.681109,0.108897,0.307474,-0.000888,0.480000,0.307474,0.101166,0.105803,0.125030,0.059181,3,0.250000,0.062432
45,2022-12-30,AI Infrastructure,28,-0.121173,-0.126578,-1.234307,-0.129611,-0.067525,0.045359,0.571429,-0.067525,-0.060124,0.136085,0.121877,0.011391,3,0.294118,-0.145365
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152,2025-03-28,Water,28,0.017320,0.013718,0.785959,0.044320,-0.328698,-0.013027,0.428571,-0.328698,-0.220986,-0.218637,-0.227469,-0.034455,3,0.857143,-0.049190
163,2025-06-27,Water,30,0.036780,0.065934,0.857222,0.050837,0.598418,0.109030,0.666667,0.598418,0.230506,0.073110,0.071444,0.016877,3,0.625000,0.269731
174,2025-09-26,Water,21,0.057145,-0.021310,-0.032357,-0.027995,-0.749384,-0.013757,0.476190,-0.749384,-0.366431,-0.193259,-0.232541,-0.025474,3,0.642857,-0.331924
185,2025-12-26,Water,9,-0.048600,-0.092375,-0.508812,-0.076133,1.850142,0.424537,0.777778,1.850142,0.737996,1.153405,0.617579,-0.074548,3,0.058824,-0.218611


In [14]:
# -------------------- 6c. Admission thresholds --------------------
# These are set from the lower tail of the time shifted placebo distance distribution.
results["distance_thresholds"].tail(30)

,rebalance_date,theme,max_relative_distance
168,2025-09-26,Cybersecurity,2.445538
169,2025-09-26,Defense,2.801113
170,2025-09-26,Fintech,2.314076
171,2025-09-26,US Infrastructure,3.702778
172,2025-09-26,Agribusiness,2.777862
173,2025-09-26,Critical Minerals,1.402182
174,2025-09-26,Water,3.068434
175,2025-09-26,Timber & Forestry,2.453725
176,2025-12-26,Robotics & AI,2.760050
177,2025-12-26,AI Infrastructure,2.166196


## Step 7 — Independent economic relevance review

In [15]:
# -------------------- 7a. Create a blinded review sample --------------------
# Score each company from 0 to 3 using filings available before the rebalance date:
# 0 = no exposure, 1 = incidental, 2 = meaningful, 3 = central business driver.
from src.evaluation_v3 import make_semantic_review_sample

semantic_sample = make_semantic_review_sample(
    latest_scores,
    metadata=None,          # optionally pass ticker indexed sector and market cap data
    top_k=10,
    controls_per_candidate=1,
    random_state=RANDOM_STATE,
)

os.makedirs("outputs", exist_ok=True)
semantic_sample["review_sheet"].to_csv(
    "outputs/semantic_review_sheet.csv", index=False
)
semantic_sample["review_key"].to_csv(
    "outputs/semantic_review_key_PRIVATE.csv", index=False
)
semantic_sample["review_sheet"].head()

,review_id,theme,ticker,relevance_score,reviewer_notes
0,R0133,Fintech,LEGH,NaN,
1,R0149,Robotics & AI,LSBK,NaN,
2,R0094,Cybersecurity,GILT,NaN,
3,R0181,US Infrastructure,GLRE,NaN,
4,R0016,AI Infrastructure,PRAA,NaN,


In [18]:
# -------------------- 7b. Evaluate a completed review --------------------
# Run after filling relevance_score in outputs/semantic_review_sheet.csv.
# Keep the private key hidden from the reviewer until scoring is complete.
#
# from src.evaluation_v3 import evaluate_semantic_review
# completed_review = pd.read_csv("outputs/semantic_review_sheet.csv")
# review_key = pd.read_csv("outputs/semantic_review_key_PRIVATE.csv")
# semantic_results = evaluate_semantic_review(completed_review, review_key)
# semantic_results

from src.evaluation_v3 import evaluate_semantic_review

completed_review = pd.read_csv("outputs/semantic_review_sheet.csv")
review_key = pd.read_csv("outputs/semantic_review_key_PRIVATE.csv")

semantic_results = evaluate_semantic_review(
    completed_review,
    review_key
)

semantic_results

,theme,candidate_precision,control_precision,enrichment,candidate_mean_relevance,control_mean_relevance,n_candidates_reviewed,n_controls_reviewed
0,Fintech,0.0,0.0,NaN,NaN,NaN,10,10
1,Robotics & AI,0.0,0.0,NaN,NaN,NaN,10,10
2,Cybersecurity,0.0,0.0,NaN,NaN,NaN,10,10
3,US Infrastructure,0.0,0.0,NaN,NaN,NaN,10,10
4,AI Infrastructure,0.0,0.0,NaN,NaN,NaN,10,10
5,Defense,0.0,0.0,NaN,NaN,NaN,10,10
6,Timber & Forestry,0.0,0.0,NaN,NaN,NaN,10,10
7,Water,0.0,0.0,NaN,NaN,NaN,10,10
8,Critical Minerals,0.0,0.0,NaN,NaN,NaN,10,10
9,Agribusiness,0.0,0.0,NaN,NaN,NaN,10,10


In [21]:
# -------------------- Step 8. Save changes to Git --------------------
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()

FILES_TO_COMMIT = [
    "ThemeCloner_V2_WalkForward_revised.ipynb",
    "src/matching_v3.py",
    "src/walkforward_v3.py",
    "src/evaluation_v3.py",
]

def run_git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
    )

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print(result.stderr)

    return result

run_git("status", "--short")
run_git("add", *FILES_TO_COMMIT)
run_git("diff", "--cached", "--stat")
run_git(
    "commit",
    "-m",
    "Add point in time theme matching and forward evaluation",
)
run_git("push")

 M ThemeCloner_V2_WalkForward.ipynb
 M outputs/fig_v2_walkforward_equity.png
?? .ipynb_checkpoints/ThemeCloner_V2_WalkForward_revised-checkpoint.ipynb
?? IMPLEMENTATION_NOTES.md
?? ThemeCloner_V2_WalkForward_revised.ipynb
?? outputs/semantic_review_key_PRIVATE.csv
?? outputs/semantic_review_sheet.csv
?? src/__pycache__/evaluation_v3.cpython-310.pyc
?? src/__pycache__/matching_v3.cpython-310.pyc
?? src/__pycache__/walkforward_v3.cpython-310.pyc
?? src/evaluation_v3.py
?? src/matching_v3.py
?? src/walkforward_v3.py


 ThemeCloner_V2_WalkForward_revised.ipynb | 2305 ++++++++++++++++++++++++++++++
 src/evaluation_v3.py                     |  414 ++++++
 src/matching_v3.py                       |  373 +++++
 src/walkforward_v3.py                    |  579 ++++++++
 4 files changed, 3671 insertions(+)

[master cc39f3c] Add point in time theme matching and forward evaluation
 4 files changed, 3671 insertions(+)
 create mode 100644 ThemeCloner_V2_WalkForward_revised.ipynb
 create mode 100644 s

CompletedProcess(args=['git', 'push'], returncode=0, stdout='', stderr='To https://github.com/AA-mini/ThemeCloner2.git\n   0359a10..cc39f3c  master -> master\n')

## Interpretation

A credible theme should show positive forward rank correlation, a positive top versus bottom exposure spread, positive basket correlation to the residualized ETF blend, and a low placebo p value. The semantic review is a separate economic check. Raw basket return remains secondary because the objective is exposure discovery rather than replication of a large cap ETF.